# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")

## 2. Data Overview
Let's review the available record sets, their `@id`s, and associated fields. Each dataset entity, such as a record set, field, or column, should always be addressed by its `@id`.

In [ ]:
# Get all record sets from the dataset
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', '')}")
    print("  Fields:")
    for fld in rs.fields:
        print(f"    - {fld.name} (@id: {fld.id}, dataType: {fld.data_type})")
    print()

For demonstration, let's display a few example records from the first available record set.

In [ ]:
# Select first record set
if record_sets:
    main_record_set = record_sets[0]
    print(f"Using record set: {main_record_set.name} (@id: {main_record_set.id})\n")

    # Show a few example records as dictionaries
    for i, rec in enumerate(dataset.records(record_set=main_record_set.id)):
        print(rec)
        if i > 2:
            break
else:
    print("No record sets found.")

## 3. Data Extraction
Extract data from the record sets of interest (using their `@id`) into pandas DataFrames for further analysis.

In [ ]:
all_record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for RecordSet '@id': {record_set_id} --> shape: {dataframes[record_set_id].shape}")

# For demonstration, show columns and sample rows for the main record set
main_rs_id = main_record_set.id if record_sets else None
if main_rs_id is not None:
    print(f"\nColumns in main record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate EDA using a numeric field from the main record set, applying standard filtering and normalization steps. All field references use their `@id` for clarity.

In [ ]:
# Select a numeric field for analysis
# We'll auto-detect a likely numeric field (e.g., MW, Abundance, etc.)
main_df = dataframes[main_rs_id]
main_fields = {f.id: f for f in main_record_set.fields}
# Heuristically pick a numeric field, e.g. MW (molecular weight) or fields annotated with Float/Integer
numeric_field_id = None
for fid, f in main_fields.items():
    if f.data_type in ('Float', 'Integer', 'Number'):
        numeric_field_id = fid
        break

if numeric_field_id is None:
    # If heuristics fail, select the first column with numeric dtype in the DataFrame
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break

print(f"Using numeric field for EDA: {numeric_field_id}")

# Drop NA and filter for values above a threshold
threshold = 10
filtered_df = main_df.dropna(subset=[numeric_field_id])
filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field, if available
# Search for a non-numeric field for grouping
group_field_id = None
for fid, f in main_fields.items():
    if f.data_type in ('Text', 'String', 'category') and fid in filtered_df.columns:
        group_field_id = fid
        break
# Fallback: pick first object dtype column not the index
if group_field_id is None:
    for col in filtered_df.columns:
        if pd.api.types.is_object_dtype(filtered_df[col]):
            group_field_id = col
            break
print(f"Grouping by field: {group_field_id}")

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot a histogram of the numeric field (after filtering)
plt.figure(figsize=(8, 4))
sns.histplot(filtered_df[numeric_field_id], bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id} (filtered > {threshold})")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If group_field_id available, boxplot
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(12, 5))
    # Limit to top 10 groups for visibility
    top_groups = filtered_df[group_field_id].value_counts().index[:10]
    sns.boxplot(data=filtered_df[filtered_df[group_field_id].isin(top_groups)],
                x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id} (top 10)")
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded dataset metadata and record sets using `mlcroissant` referencing all data structures by their `@id`.
- Explored available record sets and their fields programmatically.
- Extracted tabular data into DataFrames for flexible exploratory analysis.
- Applied filtering and normalization to numeric fields and demonstrated grouping by categorical fields.
- Visualized the distribution of key metrics and comparisons between data groups.

This workflow can be adapted to other record sets or datasets with Croissant schemas for transparent and reproducible data pipelines.